In [ ]:
import os
import re
import pandas as pd

def extract_metadata(text):
    """Extract metadata like CIK, name, form, date."""
    def search(pattern):
        m = re.search(pattern, text)
        return m.group(1).strip() if m else None

    return {
        "company_name": search(r"COMPANY CONFORMED NAME:\s+(.+)"),
        "cik": search(r"CENTRAL INDEX KEY:\s+(\d+)"),
        "form_type": search(r"CONFORMED SUBMISSION TYPE:\s+(.+)"),
        "report_date": search(r"CONFORMED PERIOD OF REPORT:\s+(\d+)")
    }

def extract_section(text, section_title):
    """Extract a specific section like MD&A or Risk Factors."""
    pattern = rf"Item\s+\d+[A-Z]?\.*\s*{section_title}.*?(?=Item\s+\d+[A-Z]?\.*|SIGNATURE|PART II|$)"
    match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
    return match.group(0).strip() if match else None

def process_full_submission(filepath, existing_keys):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
    except Exception as e:
        print(f"Could not read file: {filepath} due to {e}")
        return None

    meta = extract_metadata(content)
    print(f"Processing {meta['company_name']} - {meta['report_date']}")
    key = (meta['cik'], meta['report_date'])

    if not meta['cik'] or not meta['report_date'] or key in existing_keys:
        return None

    meta["mdna_section"] = extract_section(content, "Management's Discussion and Analysis")
    meta["risk_factors"] = extract_section(content, "Risk Factors")

    return meta

def process_all(root_dir, output_csv="10q_0_1000.csv"):
    results = []
    seen = set()

    if os.path.exists(output_csv):
        df_old = pd.read_csv(output_csv)
        seen = set(zip(df_old['cik'].astype(str), df_old['report_date'].astype(str)))
    else:
        df_old = None

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname == "full-submission.txt":
                fpath = os.path.join(dirpath, fname)
                result = process_full_submission(fpath, seen)
                if result:
                    results.append(result)
                    seen.add((result['cik'], result['report_date']))

    if results:
        df_new = pd.DataFrame(results)
        if df_old is not None:
            df_combined = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_combined = df_new
        df_combined.to_csv(output_csv, index=False)
        print(f"✅ Saved {len(results)} new entries to {output_csv}")
    else:
        print("ℹ️ No new filings found.")

# Update this to your actual path
process_all(r"C:/Users/sudet/Desktop/to-be-processed")


In [ ]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from html import unescape

def clean_extracted_text(raw_chunk):
    """Final cleaner for SEC 10-Q section text, preserving punctuation but removing HTML debris."""

    # 1. Decode HTML entities
    decoded = unescape(raw_chunk)

    # 2. Parse HTML
    try:
        soup = BeautifulSoup(decoded, "html.parser")
        cleaned = soup.get_text(separator=" ")
    except Exception:
        soup = BeautifulSoup(decoded, "html5lib")
        cleaned = soup.get_text(separator=" ")

    # 3. Normalize characters
    cleaned = cleaned.replace('\xa0', ' ')
    cleaned = cleaned.replace('’', "'").replace('“', '"').replace('”', '"')
    cleaned = cleaned.replace('\u2014', '-')  # em dash

    # 4. Lowercase
    cleaned = cleaned.lower()

    # 5. Remove junk words with "nbsp" inside
    cleaned = re.sub(r"\b\w*nbsp\w*\b", " ", cleaned)

    # 6. Remove stray ampersands or repeating table characters
    cleaned = re.sub(r"(?:\s*&\s*){2,}", " ", cleaned)  # collapse & & &
    cleaned = re.sub(r"\s*&\s*", " ", cleaned)          # remove standalone &

    # 7. Whitespace normalization
    cleaned = re.sub(r"\s+", " ", cleaned)

    return cleaned.strip()

def extract_section(raw, start_title, end_titles):
    text = raw.upper()
    text = re.sub(r"\s+", " ", text)

    start_pattern = rf"{start_title}"
    end_pattern = rf"({'|'.join(end_titles)})"
    match = re.search(start_pattern + r"(.*?)" + end_pattern, text, re.DOTALL | re.IGNORECASE)
    if not match:
        return None

    raw_chunk = match.group(1)
    return clean_extracted_text(raw_chunk)



# --- METADATA ---

def extract_metadata(text):
    """Extract metadata fields like company name, cik, etc."""
    def search(pattern):
        match = re.search(pattern, text)
        return match.group(1).strip() if match else None

    return {
        "company_name": search(r"COMPANY CONFORMED NAME:\s+(.+)"),
        "cik": search(r"CENTRAL INDEX KEY:\s+(\d+)"),
        "form_type": search(r"CONFORMED SUBMISSION TYPE:\s+(.+)"),
        "report_date": search(r"CONFORMED PERIOD OF REPORT:\s+(\d+)")
    }

# --- FILE PROCESSING ---

def process_full_submission(filepath, seen_keys):
    """Process one SEC filing."""
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            raw = f.read()
    except Exception as e:
        print(f"❌ Failed to read {filepath}: {e}")
        return None

    meta = extract_metadata(raw)
    if not meta['cik'] or not meta['report_date']:
        return None

    key = (meta['cik'], meta['report_date'])
    if key in seen_keys:
        return None

    meta["risk_factors"] = extract_section(
        raw,
        r"ITEM\s+1A[^\n]{0,40}?RISK FACTORS",
        ["ITEM 1B", "ITEM 2", "PART II", "SIGNATURE"]
    )

    meta["mdna_section"] = extract_section(
        raw,
        r"ITEM\s+2[^\n]{0,40}?MANAGEMENT.*?DISCUSSION",
        ["ITEM 3", "PART II", "SIGNATURE"]
    )

    print(f"📄 Processed {meta['company_name']} - {meta['report_date']}")
    return meta

# --- DIRECTORY WALK + MULTITHREADING ---

def process_all_files(root_dir, output_csv="output_10q.csv", max_workers=8):
    results = []
    seen = set()

    if os.path.exists(output_csv):
        df_old = pd.read_csv(output_csv)
        seen = set(zip(df_old['cik'].astype(str), df_old['report_date'].astype(str)))
    else:
        df_old = None

    filepaths = [
        os.path.join(dirpath, fname)
        for dirpath, _, files in os.walk(root_dir)
        for fname in files
        if fname.lower() == "full-submission.txt"
    ]

    print(f"📂 Found {len(filepaths)} files to process...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_full_submission, path, seen): path for path in filepaths}
        for future in as_completed(futures):
            result = future.result()
            if result:
                results.append(result)
                seen.add((result['cik'], result['report_date']))

    if results:
        df_new = pd.DataFrame(results)
        if df_old is not None:
            df_combined = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_combined = df_new
        df_combined.to_csv(output_csv, index=False)
        print(f"✅ Saved {len(results)} new filings to {output_csv}")
    else:
        print("ℹ️ No new data extracted.")

# 📌 Update path to your actual directory
process_all_files(r"C:/Users/sudet/Desktop/to-be-processed")


In [ ]:
df = pd.read_csv('output_10q.csv')

In [ ]:
df.head()

In [ ]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from html import unescape

# -------------------- CLEANING --------------------

def clean_extracted_text(raw_chunk):
    decoded = unescape(raw_chunk)
    try:
        soup = BeautifulSoup(decoded, "html.parser")
        cleaned = soup.get_text(separator=" ")
    except Exception:
        soup = BeautifulSoup(decoded, "html5lib")
        cleaned = soup.get_text(separator=" ")

    cleaned = cleaned.replace('\xa0', ' ')
    cleaned = cleaned.replace('’', "'").replace('“', '"').replace('”', '"')
    cleaned = cleaned.replace('\u2014', '-')
    cleaned = cleaned.lower()
    cleaned = re.sub(r"\b\w*nbsp\w*\b", " ", cleaned)
    cleaned = re.sub(r"(?:\s*&\s*){2,}", " ", cleaned)
    cleaned = re.sub(r"\s*&\s*", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)

    return cleaned.strip()

# -------------------- EXTRACT SECTION --------------------

def extract_section(raw, start_title, end_titles):
    text = raw.upper()
    text = re.sub(r"\s+", " ", text)
    start_pattern = rf"{start_title}"
    end_pattern = rf"({'|'.join(end_titles)})"
    match = re.search(start_pattern + r"(.*?)" + end_pattern, text, re.DOTALL | re.IGNORECASE)
    if not match:
        return None
    raw_chunk = match.group(1)
    return clean_extracted_text(raw_chunk)

# -------------------- METADATA --------------------

def extract_metadata(text):
    def search(pattern):
        match = re.search(pattern, text)
        return match.group(1).strip() if match else None

    return {
        "company_name": search(r"COMPANY CONFORMED NAME:\s+(.+)"),
        "cik": search(r"CENTRAL INDEX KEY:\s+(\d+)"),
        "form_type": search(r"CONFORMED SUBMISSION TYPE:\s+(.+)"),
        "report_date": search(r"CONFORMED PERIOD OF REPORT:\s+(\d+)")
    }

# -------------------- PROCESS FILE --------------------

write_lock = Lock()

def process_full_submission(filepath, seen_keys, output_csv):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            raw = f.read()
    except Exception as e:
        print(f"❌ Failed to read {filepath}: {e}")
        return

    meta = extract_metadata(raw)
    if not meta['cik'] or not meta['report_date']:
        return

    key = (meta['cik'], meta['report_date'])
    if key in seen_keys:
        return

    meta["risk_factors"] = extract_section(
        raw, r"ITEM\s+1A[^\n]{0,40}?RISK FACTORS", ["ITEM 1B", "ITEM 2", "PART II", "SIGNATURE"]
    )
    meta["mdna_section"] = extract_section(
        raw, r"ITEM\s+2[^\n]{0,40}?MANAGEMENT.*?DISCUSSION", ["ITEM 3", "PART II", "SIGNATURE"]
    )

    # Append row immediately to CSV
    write_lock.acquire()
    try:
        df_row = pd.DataFrame([meta])
        df_row.to_csv(output_csv, mode='a', header=not os.path.exists(output_csv), index=False)
        seen_keys.add(key)
        print(f"✅ Saved: {meta['company_name']} ({meta['report_date']})")

        # Rename file
        folder, filename = os.path.split(filepath)
        new_name = os.path.join(folder, f"processed_{filename}")
        os.rename(filepath, new_name)
    except Exception as e:
        print(f"⚠️ Error saving {filepath}: {e}")
    finally:
        write_lock.release()

# -------------------- MAIN --------------------

def process_all_files(root_dir, output_csv="10q_1000_2000.csv", max_workers=10):
    seen = set()

    # Load existing seen keys
    if os.path.exists(output_csv):
        try:
            df_old = pd.read_csv(output_csv)
            seen = set(zip(df_old['cik'].astype(str), df_old['report_date'].astype(str)))
        except Exception:
            print("⚠️ Failed to load existing output CSV. Starting fresh.")

    # Find all unprocessed files
    filepaths = [
        os.path.join(dirpath, fname)
        for dirpath, _, files in os.walk(root_dir)
        for fname in files
        if fname.lower() == "full-submission.txt"
           and not fname.startswith("processed_")
    ]

    print(f"📂 Found {len(filepaths)} files to process...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for path in filepaths:
            executor.submit(process_full_submission, path, seen, output_csv)

# -------------------- RUN --------------------

process_all_files(r"C:/Users/sudet/Desktop/to-be-processed")


In [ ]:
from bs4 import BeautifulSoup
import re
from typing import Dict

def extract_sec_sections_from_html_file(file_path: str) -> Dict[str, str]:
    """
    Extracts 'Risk Factors' and 'MD&A' sections from an SEC filing in HTML format.

    Args:
        file_path (str): Path to the HTML file.

    Returns:
        Dict[str, str]: Dictionary with 'risk' and 'mda' keys containing the corresponding section text.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        html_content = f.read()

    soup = BeautifulSoup(html_content, "html.parser")

    # Convert soup to text with line breaks to preserve formatting
    text = soup.get_text(separator="\n")

    # Normalize whitespace
    text = re.sub(r'\xa0', ' ', text)
    text = re.sub(r' +', ' ', text)

    # Patterns for section headers
    risk_start_pattern = re.compile(r'item\s*1a[\.\:\-\s]*risk factors', re.IGNORECASE)
    risk_end_pattern = re.compile(r'item\s*2[\.\:\-\s]+', re.IGNORECASE)

    mda_start_pattern = re.compile(r'item\s*2[\.\:\-\s]*management[’\'`s]* discussion and analysis', re.IGNORECASE)
    mda_end_pattern = re.compile(r'item\s*3[\.\:\-\s]+', re.IGNORECASE)

    def extract_section(start_pat, end_pat) -> str:
        starts = [m.start() for m in start_pat.finditer(text)]
        ends = [m.start() for m in end_pat.finditer(text)]
        if not starts or not ends:
            return ""
        for s in starts:
            for e in ends:
                if s < e:
                    return text[s:e].strip()
        return ""

    return {
        "risk": extract_section(risk_start_pattern, risk_end_pattern),
        "mda": extract_section(mda_start_pattern, mda_end_pattern)
    }


In [ ]:
sections = extract_sec_sections_from_html_file("full-submission.txt")

print("=== RISK FACTORS ===\n", sections["risk"])  # preview
print("\n=== MD&A ===\n", sections["mda"])         # preview


In [ ]:
import re
import os
from collections import defaultdict

class SEC10QFileParser:
    def __init__(self):
        # Define patterns for section headers (case insensitive)
        self.risk_factor_patterns = [
            r'item\s*1a\.?\s*risk\s*factors',
            r'risk\s*factors',
            r'item\s*2\.?\s*risk\s*factors',
            r'item\s*1a\s*risk\s*factors',
            r'item\s*1a\.\s*risk\s*factors'
        ]
        
        self.mda_patterns = [
            r'item\s*2\.?\s*management\'?s\s*discussion\s*and\s*analysis',
            r'management\'?s\s*discussion\s*and\s*analysis',
            r'md\s*&\s*a',
            r'mda',
            r'item\s*7\.?\s*management\'?s\s*discussion\s*and\s*analysis',
            r'item\s*2\s*management\'?s\s*discussion\s*and\s*analysis'
        ]
        
        self.section_end_patterns = [
            r'item\s*[1-9][a-z]?\.?\s*[a-z]',
            r'item\s*[1-9][a-z]?\.?\s*[A-Z]',
            r'^\s*[A-Z][A-Z\s]+\s*$',
            r'^\s*item\s*[1-9][a-z]?\s*$'
        ]
        
        self.min_section_length = 200  # Minimum characters to consider as a valid section
        
    def load_text_file(self, file_path):
        """Load text from a local file with proper encoding handling"""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
            # If utf-8 fails, try other common encodings
        except UnicodeDecodeError:
            try:
                with open(file_path, 'r', encoding='latin-1') as f:
                    text = f.read()
            except Exception as e:
                print(f"Error reading file {file_path}: {e}")
                return None
        
        # Normalize whitespace and clean up
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[\xa0\u200b]', ' ', text)  # Remove non-breaking spaces and zero-width spaces
        return text
    
    def split_into_sections(self, text):
        """Split text into potential sections using common patterns"""
        # First try splitting on ITEM headers
        sections = re.split(r'(\n\s*ITEM\s*[0-9][A-Z]?\s*[.:-]?\s*)', text, flags=re.IGNORECASE)
        
        # If that didn't produce meaningful sections, try other patterns
        if len(sections) < 5:
            sections = re.split(r'(\n\s*[A-Z][A-Z\s]+\s*\n)', text)
        
        # Recombine the split parts with their headers
        combined_sections = []
        for i in range(1, len(sections), 2):
            combined_sections.append(sections[i] + sections[i+1])
        
        # If we still don't have good sections, fall back to double newline splits
        if len(combined_sections) < 3:
            combined_sections = re.split(r'\n{2,}', text)
        
        return combined_sections
    
    def is_section_header(self, text, patterns):
        """Check if text matches any of the header patterns"""
        text = text.lower().strip()
        for pattern in patterns:
            if re.fullmatch(pattern, text, flags=re.IGNORECASE):
                return True
        return False
    
    def find_section(self, sections, patterns):
        """Find the most likely section matching the patterns"""
        # First pass: look for exact header matches
        for i, section in enumerate(sections):
            # Split into first line (potential header) and rest
            lines = section.split('\n', 1)
            if len(lines) > 0 and self.is_section_header(lines[0], patterns):
                return '\n'.join(lines[1:]) if len(lines) > 1 else ""
        
        # Second pass: look for pattern in first few lines
        for section in sections:
            lines = section.split('\n')
            for line in lines[:5]:  # Check first 5 lines
                if self.is_section_header(line, patterns):
                    start_idx = section.find(line) + len(line)
                    return section[start_idx:].strip()
        
        # Third pass: look for keywords in content
        keyword_patterns = [
            r'risk\s*factor', 
            r'could\s*adversely\s*affect',
            r'may\s*affect',
            r'forward-looking',
            r'uncertainties',
            r'potential\s*risks',
            r'could\s*negatively\s*impact'
        ] if patterns == self.risk_factor_patterns else [
            r'discussion\s*and\s*analysis',
            r'financial\s*condition',
            r'results\s*of\s*operations',
            r'liquidity',
            r'capital\s*resources',
            r'revenue\s*recognition',
            r'operating\s*results',
            r'gross\s*margin'
        ]
        
        keyword_sections = []
        for section in sections:
            content = section.lower()
            matches = sum(1 for pattern in keyword_patterns if re.search(pattern, content))
            if matches >= 2:  # At least two keyword matches
                keyword_sections.append((section, matches))
        
        if keyword_sections:
            # Return the section with the most keyword matches
            keyword_sections.sort(key=lambda x: x[1], reverse=True)
            return keyword_sections[0][0]
        
        return None
    
    def clean_section(self, section):
        """Clean up a section by removing unwanted content"""
        if not section:
            return ""
        
        # Remove page numbers and headers/footers
        section = re.sub(r'\n\s*\d+\s*\n', '\n', section)
        section = re.sub(r'-\s*\d+\s*-', '', section)
        
        # Remove table of contents if present
        toc_pattern = r'\(?continued\)?\s*\.+\s*\d+'
        section = re.sub(toc_pattern, '', section, flags=re.IGNORECASE)
        
        # Remove multiple blank lines
        section = re.sub(r'\n{3,}', '\n\n', section)
        
        # Remove excessive whitespace
        section = re.sub(r'[ \t]{2,}', ' ', section)
        
        return section.strip()
    
    def extract_risk_factors(self, text):
        """Extract risk factors section"""
        sections = self.split_into_sections(text)
        risk_section = self.find_section(sections, self.risk_factor_patterns)
        return self.clean_section(risk_section)
    
    def extract_mda(self, text):
        """Extract MD&A section"""
        sections = self.split_into_sections(text)
        mda_section = self.find_section(sections, self.mda_patterns)
        return self.clean_section(mda_section)
    
    def parse_10q_file(self, file_path):
        """Parse a 10-Q filing from a local text file"""
        print(f"Processing file: {file_path}")
        text = self.load_text_file(file_path)
        if not text:
            return None
        
        results = {
            'file_path': file_path,
            'risk_factors': self.extract_risk_factors(text),
            'mda': self.extract_mda(text),
            'raw_text': text  # Optional - can be large
        }
        
        return results
    
    def parse_directory(self, directory_path):
        """Parse all .txt files in a directory"""
        results = []
        for filename in os.listdir(directory_path):
            if filename.lower().endswith('.txt'):
                file_path = os.path.join(directory_path, filename)
                result = self.parse_10q_file(file_path)
                if result:
                    results.append(result)
        return results

# Example usage
if __name__ == "__main__":
    parser = SEC10QFileParser()
    
    # Parse a single file
    file_path = "full-submission.txt"
    results = parser.parse_10q_file(file_path)
    
    if results:
        print("\n=== Risk Factors ===")
        print(results['risk_factors'][:2000] + "...")  # Print first 2000 chars
        
        print("\n=== MD&A ===")
        print(results['mda'][:2000] + "...")
        
        # Save results to files
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        
        with open(f'{base_name}_risk_factors.txt', 'w', encoding='utf-8') as f:
            f.write(results['risk_factors'])
        
        with open(f'{base_name}_mda.txt', 'w', encoding='utf-8') as f:
            f.write(results['mda'])
    
    # Alternatively, parse a whole directory
    # directory_path = "path/to/your/10q_files/"
    # all_results = parser.parse_directory(directory_path)
    # for result in all_results:
    #     print(f"\nProcessed: {result['file_path']}")
    #     print(f"Risk factors length: {len(result['risk_factors'])}")
    #     print(f"MD&A length: {len(result['mda'])}")

In [ ]:
import re
import html
from typing import Dict, Optional


class SECFilingParser:
    def __init__(self):
        self.section_patterns = {
            "risk_factors": [
                r"item\s*1a[\.\-]?\s*risk\s*factors?",
                r"risk\s*factors?"
            ],
            "mda": [
                r"item\s*2[\.\-]?\s*management[’']?s\s*discussion\s*and\s*analysis",
                r"management[’']?s\s*discussion\s*and\s*analysis.*?results\s*of\s*operations"
            ]
        }
        self.item_boundary = re.compile(r"item\s*\d+[a-z]?\s*[\.\:\-]?", re.IGNORECASE)

    def extract_section(self, text: str, section_key: str) -> Optional[str]:
        patterns = self.section_patterns.get(section_key, [])
        start_match = None

        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                start_match = match
                break

        if not start_match:
            return None

        # Find next section boundary
        start_idx = start_match.start()
        end_match = self.item_boundary.search(text[start_idx + 20:])
        end_idx = start_idx + 20 + end_match.start() if end_match else len(text)

        section_text = text[start_idx:end_idx]
        return self.clean_text(section_text)

    def clean_text(self, raw: str) -> str:
        text = html.unescape(raw)
        text = re.sub(r"<[^>]+>", " ", text)  # remove HTML tags
        text = re.sub(r"\s+", " ", text)  # normalize whitespace
        return text.strip()

    def parse(self, text: str) -> Dict[str, Optional[str]]:
        return {
            "risk_factors": self.extract_section(text, "risk_factors"),
            "mda": self.extract_section(text, "mda")
        }


# Example usage
if __name__ == "__main__":
    with open("full-submission.txt", encoding="utf-8") as f:
        filing_text = f.read()

    parser = SECFilingParser()
    sections = parser.parse(filing_text)

    for sec, content in sections.items():
        print(f"\n--- {sec.upper()} ---\n")
        print(content[:1000] + "..." if content else "Section not found.")


In [ ]:
import re
import re
from bs4 import BeautifulSoup
import html

def split_10q_sections(text: str) -> dict[str, str]:
    """
    Splits a 10-Q filing into sections based on 'Item X.' markers.
    
    Args:
        text (str): Raw text of a 10-Q filing.

    Returns:
        dict[str, str]: A dictionary where keys are section headers (e.g. "Item 1.") 
                        and values are the corresponding section content.
    """
    # Find all matches of "Item X." headers
    pattern = re.compile(r"(item\s+\d+[a-z]?\.?)", re.IGNORECASE)
    matches = list(pattern.finditer(text))

    if not matches:
        return {}

    sections = {}
    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        header = matches[i].group().strip()
        content = text[start:end].strip()
        sections[header] = content

    return sections

# def extract_key_sections(sections: dict[str, str]) -> dict[str, str]:
#     """
#     Identifies and extracts Risk Factors and MD&A sections from a 10-Q section dictionary.
#
#     Args:
#         sections (dict): Dictionary mapping section headers (e.g., "Item 1A.") to content.
#
#     Returns:
#         dict: Dictionary with 'mda' and 'risk_factors' keys and their matching content.
#     """
#     key_sections = {"mda": None, "risk_factors": None}
#
#     for header, content in sections.items():
#         # Combine header and first 300 chars of content for flexible searching
#         target = f"{header.lower()} {content[:300].lower()}"
#
#         if key_sections["mda"] is None and any(kw in target for kw in [
#             "management", "discussion", "results of operations", "mda"
#         ]):
#             key_sections["mda"] = content.strip()
#
#         elif key_sections["risk_factors"] is None and any(kw in target for kw in [
#             "risk factors", "risk", "risks"
#         ]):
#             key_sections["risk_factors"] = content.strip()
#
#         # Stop if both found
#         if all(key_sections.values()):
#             break
#
#     return key_sections

def extract_key_sections(sections: dict[str, str]) -> dict[str, str]:
    def is_probably_mda(text: str) -> bool:
        text = text.lower()
        return any(phrase in text for phrase in [
            "management’s discussion and analysis",
            "management discussion and analysis",
            "discussion and analysis of financial condition",
            "results of operations"
        ])

    def is_probably_risk(text: str) -> bool:
        text = text.lower()
        return any(phrase in text for phrase in [
            "risk factors", "risks related", "risks we face", "you should consider carefully the following"
        ])

    key_sections = {"mda": None, "risk_factors": None}

    # 1️⃣ Try Item 2 for MD&A
    for header, content in sections.items():
        if "item 2" in header.lower():
            if is_probably_mda(content):
                key_sections["mda"] = content.strip()
            break  # Try item 2 first and break either way

    # 2️⃣ Try Item 1A for Risk Factors
    for header, content in sections.items():
        if "item 1a" in header.lower():
            if is_probably_risk(content):
                key_sections["risk_factors"] = content.strip()
            break

    # 3️⃣ Fallback: scan all sections if not found
    for header, content in sections.items():
        lc = (header + " " + content[:300]).lower()
        if key_sections["mda"] is None and is_probably_mda(lc):
            key_sections["mda"] = content.strip()
        if key_sections["risk_factors"] is None and is_probably_risk(lc):
            key_sections["risk_factors"] = content.strip()
        if all(key_sections.values()):
            break

    return key_sections


def clean_section_text(text: str) -> str:
    """
    Fully clean a section from SEC filings:
    - Unescape HTML entities (e.g., &#8220;)
    - Strip all HTML tags
    - Normalize whitespace and remove layout artifacts
    """
    if not text:
        return ""

    # Decode HTML entities (e.g. &#8217;)
    text = html.unescape(text)

    # Parse and remove HTML tags
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text(separator=" ")

    # Remove any remaining artifacts like table padding or font styling
    text = re.sub(r"\s+", " ", text)                  # collapse all whitespace
    text = re.sub(r"\s([.,;:!?])", r"\1", text)        # fix spaces before punctuation
    text = re.sub(r"(table of contents|<page>|continued\s*on\s*next\s*page)", "", text, flags=re.I)
    return text.strip()


In [ ]:
with open("full-submission_2.txt", encoding="utf-8") as f:
    filing_text = f.read()

sections = split_10q_sections(filing_text)

# Preview the extracted sections
for header, content in sections.items():
     print(f"\n--- {header.upper()} ---\n")
     print(content[:500], "...\n")
key_parts = extract_key_sections(sections)

print("--- RISK FACTORS ---\n")
print(key_parts["risk_factors"][:1500] if key_parts["risk_factors"] else "Not found.")

print("\n--- MANAGEMENT DISCUSSION & ANALYSIS ---\n")
print(key_parts["mda"][:1500] if key_parts["mda"] else "Not found.")

cleaned_risks = clean_section_text(key_parts["risk_factors"])
cleaned_mda = clean_section_text(key_parts["mda"])

print("--- MDA ---\n", cleaned_mda[:1000])
print("\n--- RISK FACTORS ---\n", cleaned_risks)




In [ ]:
with open("processed_full-submission.txt", encoding="utf-8") as f:
    filing_text = f.read()

sections = split_10q_sections(filing_text)

# Preview the extracted sections
for header, content in sections.items():
    print(f"\n--- {header.upper()} ---\n")
    print(content[:500], "...\n")
    
key_parts = extract_key_sections(sections)

print("--- RISK FACTORS ---\n")
print(key_parts["risk_factors"][:1500] if key_parts["risk_factors"] else "Not found.")

print("\n--- MANAGEMENT DISCUSSION & ANALYSIS ---\n")
print(key_parts["mda"][:1500] if key_parts["mda"] else "Not found.")


In [ ]:
# Clean the entire full-submission using BeautifulSoup and return plain text version
with open("full-submission_2.txt", encoding="utf-8") as f:
    text = f.read()
    
    
def clean_filing_text(text):
    soup = BeautifulSoup(text, "lxml")  
    clean_text = soup.get_text(separator=" ")  
    clean_text = re.sub(r"\s+", " ", clean_text).strip()  
    return clean_text

clean_filing_text(text)

In [ ]:
from bs4 import BeautifulSoup, Tag
import re
from typing import List, Dict

SECTION_LABELS = {
    "item 1a": "risk_factors",
    "item 2": "management_discussion",
}


SECTION_START_RE = re.compile(r"item\s*(1a|2)\.?\s*", re.IGNORECASE)
SECTION_END_RE = re.compile(r"item\s*(2|3)\.?\s*", re.IGNORECASE)


def build_hierarchy(soup: BeautifulSoup) -> List[Tag]:
    """Extract relevant div/span/p blocks with their tag-level text."""
    tags = []
    for tag in soup.find_all(["div", "p", "span"]):
        text = tag.get_text(separator=" ", strip=True)
        if not text:
            continue
        tags.append(tag)
    return tags


def extract_sections_by_heading(soup: BeautifulSoup) -> Dict[str, str]:
    blocks = build_hierarchy(soup)
    extracted = {}
    
    i = 0
    while i < len(blocks):
        text = blocks[i].get_text(separator=" ", strip=True).lower()

        match = SECTION_START_RE.match(text)
        if match:
            label = SECTION_LABELS.get(f"item {match.group(1).lower()}")
            if not label:
                i += 1
                continue

            start = i
            # Search forward for end
            for j in range(i + 1, len(blocks)):
                next_text = blocks[j].get_text(separator=" ", strip=True).lower()
                if SECTION_END_RE.match(next_text):
                    break
            else:
                j = len(blocks)

            content = "\n".join(block.get_text(separator=" ", strip=True) for block in blocks[start:j])
            extracted[label] = content
            i = j
        else:
            i += 1

    return extracted


In [ ]:
with open("full-submission_2.txt", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "lxml")
sections = extract_sections_by_heading(soup)


print("MDA:\n", sections["management_discussion"])
print("\nRisk Factors:\n", sections["risk_factors"])
